# Colab AI Studio — Worker Distant GPU (Tesla T4 16 Go)

Ce notebook configure et démarre le worker distant à haute performance pour **Colab AI Studio**.

Il assure :
1. La vérification du GPU NVIDIA Tesla T4 disponible sur Google Colab.
2. L'installation des bibliothèques nécessaires (`FastAPI`, `Uvicorn`, `PyNgrok`, `torch`, `Pydantic`).
3. Le démarrage de l'automate `ModelSwapper` assurant la permutation à chaud en VRAM sans saturation OOM.
4. L'ouverture d'un tunnel chiffré (Ngrok / Cloudflare) fournissant une URL publique à copier dans l'interface de **Colab AI Studio**.

In [ ]:
# Étape 1 : Diagnostic GPU NVIDIA
!nvidia-smi

In [ ]:
# Étape 2 : Installation des dépendances du Worker
!pip install -q fastapi uvicorn pyngrok pydantic httpx psutil

In [ ]:
# Étape 3 : Implémentation du ModelSwapper et de l'API Worker Colab
import sys
import os
from pathlib import Path

# Importer l'automate et le serveur FastAPI
from app.colab_worker.model_swapper import ModelSwapper
from app.colab_worker.server import create_colab_worker_app

swapper = ModelSwapper()
app = create_colab_worker_app(swapper=swapper, gpu_name="NVIDIA Tesla T4 (Colab)", vram_free_mb=14800)
print("✓ Worker Colab initialisé avec succès.")

In [ ]:
# Étape 4 : Ouverture du tunnel public (Ngrok / Cloudflare)
from pyngrok import ngrok

# Spécifiez votre token Ngrok si disponible (ou exécutez directement en mode public)
NGROK_TOKEN = os.getenv("NGROK_TOKEN", "")
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("=" * 60)
print("🚀 COLAB AI STUDIO WORKER ACTIF !")
print(f"Copiez cette URL dans Colab AI Studio : {public_url}")
print("=" * 60)

In [ ]:
# Étape 5 : Lancement du serveur Uvicorn
import uvicorn
uvicorn.run(app, host="0.0.0.0", port=8000)